# YOLO VERSION 12 X

## Import Library

In [ ]:
import torch 
import torch.nn as nn
import math 
from ultralytics import YOLO
import torch.nn.functional as F
import numpy as np 
from torch.utils.data import DataLoader, Dataset
import pandas as pd 
import matplotlib.pyplot as plt 
import torch.optim as optim 
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import cv2
from typing import Any
from types import SimpleNamespace
import copy
import torchvision

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

## Block

In [22]:
def autopad(k, p=None, d=1):
    if d > 1:
        k = d * (k - 1) + 1
    if p is None:
        p = k // 2
    return p

class Conv(nn.Module): 
    def __init__(self, c1, c2, k=1, s=1, p=None, g=1, d=1, act=True):
        super().__init__()
        self.conv   = nn.Conv2d(c1, c2, k, s, autopad(k, p, d), groups=g, dilation=d, bias=False) 
        self.bn     = nn.BatchNorm2d(c2, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        self.act    =  nn.SiLU(inplace=True) if act else nn.Identity()
        
    def forward (self, x):
        return self.act(self.bn(self.conv(x)))

class ConvPE(nn.Module): 
    def __init__(self, c1, c2, k=1, s=1, p=None, g=1, d=1, act=True):
        super().__init__()
        self.conv   = nn.Conv2d(c1, c2, k, s, autopad(k, p, d), groups=g, dilation=d) 
        self.bn     = nn.BatchNorm2d(c2, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        self.act    =  nn.SiLU(inplace=True) if act else nn.Identity()
        
    def forward (self, x):
        return self.act(self.bn(self.conv(x)))

class Bottleneck(nn.Module) : 
    def __init__(self, c1, c2 ,shortcut=True, k=3, e=0.5):
        super().__init__()
        c_ = int(e*c2)
        self.cv1 = Conv(c1, c_, k, 1)
        self.cv2 = Conv(c_, c2, k, 1)
        self.add = shortcut and c1 == c2    
        
    def forward (self, x):
        return x + self.cv2(self.cv1(x)) if self.add else self.cv2(self.cv1(x))  


class C3k(nn.Module):
    def __init__(self, c1, c2, n=2, shortcut=True, g=1):
        super().__init__()

        c_ = c2 // 2

        self.cv1 = Conv(c1, c_, 1, 1)
        self.cv2 = Conv(c1, c_, 1, 1)
        self.cv3 = Conv(2 * c_, c2, 1)

        self.m = nn.Sequential(
            *[Bottleneck(c_, c_, shortcut, k=3, e=1.0) for _ in range(n)]
        )

    def forward(self, x):
        return self.cv3(torch.cat((self.m(self.cv1(x)), self.cv2(x)), 1))

class Attention12(nn.Module):
    def __init__(self, c):
        super().__init__()

        self.qkv = Conv(c, c * 3, 1, act=False)

        self.proj = Conv(c, c, 1, act=False)
        #self.pe = nn.Conv2d(c,c,7,1,3,groups=c)
        self.pe = ConvPE(c, c, 7, 1, g=c, act=False)

    def forward(self, x):

        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, 1)

        attn = (q @ k.transpose(-2, -1))
        attn = attn.softmax(-1)

        x = attn @ v

        x = self.proj(x)

        x = x + self.pe(v)

        return x

class ABlock(nn.Module):

    def __init__(self, c):
        super().__init__()

        self.attn = Attention12(c)

        hidden = int(c * 1.2)

        self.mlp = nn.Sequential(
            Conv(c, hidden, 1),
            Conv(hidden, c, 1, act=False)
        )

    def forward(self, x):

        x = x + self.attn(x)
        x = x + self.mlp(x)

        return x

class A2C2f(nn.Module):
    def __init__(self, c1, c2, n=2, block_type="attn"):
        super().__init__()

        c_ = c2 // 2

        self.cv1 = Conv(c1, c_, 1)
        self.cv2 = Conv((n + 1) * c_, c2, 1)

        if block_type == "attn":
            self.m = nn.ModuleList(
                nn.Sequential(ABlock(c_), ABlock(c_)) for _ in range(n)
            )
            self.gamma = nn.Parameter(torch.ones(c2))
            
        elif block_type == "c3k":
            self.m = nn.ModuleList(
                C3k(c_, c_) for _ in range(n)
            )
            self.gamma = None

        self.add = c1 == c2

    def forward(self, x):
        shortcut = x

        x = self.cv1(x)

        y = [x]
        for m in self.m:
            x = m(x)
            y.append(x)

        out = self.cv2(torch.cat(y, 1))

        if self.add:
            out = shortcut + self.gamma.view(1, -1, 1, 1) * out

        return out

class C3K2V12(nn.Module):

    def __init__(self, c1, c2, n=2):
        super().__init__()

        c_ = c1 // 2

        self.cv1 = Conv(c1, c1, 1)   
        self.cv2 = Conv((2 + n) * c_, c2, 1)

        self.m = nn.ModuleList(
            [C3k(c_, c_, 2) for _ in range(n)]
        )

    def forward(self, x):

        x = self.cv1(x)

        x1, x2 = x.chunk(2,1)

        y = [x1, x2]

        for m in self.m:
            x2 = m(x2)
            y.append(x2)

        return self.cv2(torch.cat(y,1))

class DWConv(Conv):
    def __init__(self, c1, c2, k=1, s=1, d=1, act=True):
        super().__init__(c1, c2, k, s, g=math.gcd(c1, c2), d=d, act=act)    

class DFL(nn.Module):
    def __init__(self, reg_max=16):
        super().__init__()
        self.reg_max = reg_max
        self.conv = nn.Conv2d(reg_max, 1, 1, bias=False)
        self.conv.weight.data[:] = torch.arange(reg_max).view(1, reg_max, 1, 1)
        self.conv.weight.requires_grad = False

    def forward(self, x):
        B, C, H, W = x.shape
        x = x.view(B, 4, self.reg_max, H, W)
        x = x.softmax(2)
        x = x.view(B * 4, self.reg_max, H, W)
        x = self.conv(x)
        return x.view(B, 4, H, W)
    
class C3K2_Neck(nn.Module):
    def __init__(self, c1, c2, n=2):
        super().__init__()

        c_ = c2 // 2  

        self.cv1 = Conv(c1, c2, 1) 
        self.cv2 = Conv((2 + n) * c_, c2, 1)

        self.m = nn.ModuleList(
            C3k(c_, c_) for _ in range(n)
        )

    def forward(self, x):
        x = self.cv1(x)

        x1, x2 = x.chunk(2, 1)

        y = [x1, x2]

        for m in self.m:
            x2 = m(x2)
            y.append(x2)

        return self.cv2(torch.cat(y, 1))


class Concat(nn.Module):
    def __init__(self, dim=1):
        super().__init__()
        self.d = dim

    def forward(self, x):
        return torch.cat(x, self.d)

## Backbone

In [ ]:
class YOLOv12Backbone(nn.Module):

    def __init__(self):
        super().__init__()

        base_c = [96, 192, 384, 768, 768]
        c1, c2, c3, c4, c5 = base_c

        # layer 0
        self.stem = Conv(3, c1, 3, 2)

        # layer 1-2
        self.dark2 = nn.Sequential(
            Conv(c1, c2, 3, 2),
            C3K2V12(c2, c3, n=2)
        )

        # layer 3-4
        self.dark3 = nn.Sequential(
            Conv(c3, c3, 3, 2),
            C3K2V12(c3, c4, n=2)
        )

        # layer 5-6
        self.dark4 = nn.Sequential(
            Conv(c4, c4, 3, 2),
            A2C2f(c4, c4, n=4, block_type="attn")
        )

        # layer 7-8
        self.dark5 = nn.Sequential(
            Conv(c4, c4, 3, 2),
            A2C2f(c4, c4, n=4, block_type="attn")
        )

        # karena p3, p4, p5 diambil dari dark3, dark4, dark5
        self.out_channels = (c4, c4, c4)

    def forward(self, x):
        x = self.stem(x)

        x = self.dark2(x)

        x = self.dark3(x)
        p3 = x   # 768 channel

        x = self.dark4(x)
        p4 = x   # 768 channel

        x = self.dark5(x)
        p5 = x   # 768 channel

        return p3, p4, p5

## Neck

In [ ]:
class YOLOv12Neck(nn.Module):
    def __init__(self, ch=(768, 768, 768)):
        super().__init__()

        p3_ch, p4_ch, p5_ch = ch

        self.up = nn.Upsample(scale_factor=2.0, mode="nearest")

        # layer 11
        # concat: p5 upsample + p4 = 768 + 768 = 1536
        self.fpn1 = A2C2f(
            p5_ch + p4_ch,   # 1536
            p4_ch,           # 768
            n=2,
            block_type="c3k"
        )

        # layer 14
        # concat: fpn1 upsample + p3 = 768 + 768 = 1536
        self.fpn2 = A2C2f(
            p4_ch + p3_ch,   # 1536
            384,             # output layer 14
            n=2,
            block_type="c3k"
        )

        # layer 15
        self.down1 = Conv(384, 384, 3, 2)

        # layer 17
        # concat: down1 + fpn1 = 384 + 768 = 1152
        self.pan1 = A2C2f(
            384 + p4_ch,     # 1152
            p4_ch,           # 768
            n=2,
            block_type="c3k"
        )

        # layer 18
        self.down2 = Conv(p4_ch, p5_ch, 3, 2)

        # layer 20
        # concat: down2 + p5 = 768 + 768 = 1536
        self.pan2 = C3K2_Neck(
            p5_ch + p5_ch,   # 1536
            p5_ch,           # 768
            n=2
        )

        self.out_channels = (384, 768, 768)

    def forward(self, p3, p4, p5):
        # layer 9-11
        x = self.up(p5)
        x = torch.cat([x, p4], dim=1)
        n4 = self.fpn1(x)

        # layer 12-14
        x = self.up(n4)
        x = torch.cat([x, p3], dim=1)
        n3 = self.fpn2(x)

        # layer 15-17
        x = self.down1(n3)
        x = torch.cat([x, n4], dim=1)
        n4 = self.pan1(x)

        # layer 18-20
        x = self.down2(n4)
        x = torch.cat([x, p5], dim=1)
        n5 = self.pan2(x)

        return n3, n4, n5

## Head

In [ ]:
class YOLOv12Head(nn.Module):
    def __init__(self, ch, nc=80, reg_max=16):
        super().__init__()

        self.nc = nc
        self.reg_max = reg_max
        self.no = nc + 4 * reg_max

        self.cv2 = nn.ModuleList()  
        self.cv3 = nn.ModuleList()  

        for i, c in enumerate(ch):
            c2 = min(c, 96)

            self.cv2.append(
                nn.Sequential(
                    Conv(c, c2, 3),
                    Conv(c2, c2, 3),
                    nn.Conv2d(c2, 4 * reg_max, 1)
                )
            )

            if i == 0:
                c_mid = c
            else:
                c_mid = c // 2

            self.cv3.append(
                nn.Sequential(
                    nn.Sequential(
                        DWConv(c, c, 3),
                        Conv(c, c_mid, 1, act=False)
                    ),
                    nn.Sequential(
                        DWConv(c_mid, c_mid, 3),
                        Conv(c_mid, c_mid, 1, act=False)
                    ),
                    nn.Conv2d(c_mid, nc, 1)
                )
            )

        self.dfl = DFL(reg_max)

    def forward(self, feats):
        outputs = []

        for i, x in enumerate(feats):
            reg = self.cv2[i](x)
            cls = self.cv3[i](x)

            out = torch.cat([reg, cls], 1)
            outputs.append(out)

        return outputs

## Full

In [ ]:
class TemperatureClsHead(nn.Module):
    def __init__(self, in_channels, num_temp_classes=3, hidden_channels=320, dropout=0.1):
        super().__init__()

        # (640, 40, 40) => (640, 40, 40)
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, hidden_channels, kernel_size=1, stride=1, bias=False),
            nn.BatchNorm2d(hidden_channels),
            nn.SiLU(inplace=True)
        )    

        self.pool = nn.AdaptiveAvgPool2d(20)
        self.drop = nn.Dropout(p=dropout, inplace=True)
        self.linear = nn.Linear(hidden_channels*20*20, num_temp_classes)

    def forward(self, x):
        x = self.conv(x)  # (640, 40, 40) => (320, 40, 40)
        x = self.pool(x) # (320, 40, 40) => (320, 20, 20)         
        x = torch.flatten(x, 1) # (B,320, 20, 20) => (B,128000)  
        x = self.drop(x)  # (B,128000) => (B,128000)
        x = self.linear(x) # (B,128000) => (B,3)       
        return x

In [ ]:
class YOLOv12(nn.Module):
    def __init__(self, nc=3, reg_max=16, num_temp_classes=3):
        super().__init__()
        
        self.num_classes = nc
        self.nc = nc
        self.reg_max = reg_max
        self.no = nc + reg_max * 4
        self.stride = torch.tensor([8., 16., 32.])

        self.backbone = YOLOv12Backbone()
        self.neck = YOLOv12Neck(self.backbone.out_channels)
        self.head = YOLOv12Head(self.neck.out_channels,nc)
        
        self.temp_head = TemperatureClsHead(
            in_channels=self.neck.out_channels[1],  # 640   FEATURE KEDUA
            num_temp_classes=num_temp_classes, # 3
            hidden_channels=320,
            dropout=0.1
        )


    def forward(self, x):
        
        p3, p4, p5 = self.backbone(x)
        p3, p4, p5 = self.neck(p3, p4, p5)

        det_out = self.head([p3, p4, p5])

        temp_logits = self.temp_head(p4)

        return det_out, temp_logits

In [ ]:
class YOLOTemperatureDataset(Dataset):
    def __init__(self, image_dir, label_dir, csv_path, img_size=640):
        self.image_dir = Path(image_dir)
        self.label_dir = Path(label_dir)
        self.csv_path = Path(csv_path)
        self.img_size = img_size

        self.df = pd.read_csv(self.csv_path)

        self.label_map = {
            "Sejuk": 0,
            "Sedang": 1,
            "Dingin": 2
        }

    def __len__(self):
        return len(self.df)

    # Ambil sampel dari dataset berdasarkan indeks
    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_name = str(row["NamaFile"]).strip()
        temp_label_text = str(row["Label"]).strip()

        if temp_label_text not in self.label_map:
            raise ValueError(f"Label tidak dikenal: {temp_label_text}")

        temp_label = self.label_map[temp_label_text]

        img_path = self.image_dir / img_name
        label_path = self.label_dir / (Path(img_name).stem + ".txt")

        if not img_path.exists():
            raise FileNotFoundError(f"Image tidak ditemukan: {img_path}")

        img = cv2.imread(str(img_path))
        if img is None:
            raise FileNotFoundError(f"Gagal membaca image: {img_path}")

        # BGR ke RGB
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Membaca Bounding Box
        boxes = []
        if label_path.exists():
            with open(label_path, "r") as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        cls, xc, yc, w, h = map(float, parts)
                        boxes.append([cls, xc, yc, w, h])

        if len(boxes) > 0:
            boxes = np.array(boxes, dtype=np.float32)
        else:
            boxes = np.zeros((0, 5), dtype=np.float32)

        # Konversi wajib untuk input model
        img = torch.from_numpy(img).permute(2, 0, 1).contiguous().float() / 255.0
        boxes = torch.tensor(boxes, dtype=torch.float32)
        temp_label = torch.tensor(temp_label, dtype=torch.long)

        return img, boxes, temp_label

In [ ]:
train_dataset = YOLOTemperatureDataset(
    image_dir="Dataset_rapih/images/train",
    label_dir="Dataset_rapih/labels/train",
    csv_path="Dataset_rapih/Informasi Dataset - TRAINING.csv",
    img_size=640
)

val_dataset = YOLOTemperatureDataset(
    image_dir="Dataset_rapih/images/valid",
    label_dir="Dataset_rapih/labels/valid",
    csv_path="Dataset_rapih/Informasi Dataset - VALIDATION.csv",
    img_size=640
)

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
def bbox_ciou(box1, box2, eps=1e-7):
    """
    Menghitung CIoU antara dua bounding box.
    
    box1.shape = [N, 4]
    box2.shape = [N, 4]

    N = jumlah bbox
    4 = [x1, y1, x2, y2]

    output:
    ciou.shape = [N, 1]
    
    box1    : prediksi 
    box2    : ground truth
    x1, y1  : kiri atas 
    x2, y2  : kanan bawah 
    """

    # Pecah koordinat bbox
    b1_x1, b1_y1, b1_x2, b1_y2 = box1.chunk(4, dim=-1) 
    b2_x1, b2_y1, b2_x2, b2_y2 = box2.chunk(4, dim=-1)

    # Lebar dan tinggi masing-masing bbox
    w1 = (b1_x2 - b1_x1).clamp(min=eps)
    h1 = (b1_y2 - b1_y1).clamp(min=eps)

    w2 = (b2_x2 - b2_x1).clamp(min=eps)
    h2 = (b2_y2 - b2_y1).clamp(min=eps)

    # Luas intersection / irisan
    inter_w = (torch.min(b1_x2, b2_x2) - torch.max(b1_x1, b2_x1)).clamp(min=0)
    inter_h = (torch.min(b1_y2, b2_y2) - torch.max(b1_y1, b2_y1)).clamp(min=0)
    inter_area = inter_w * inter_h

    # Luas union / gabungan
    area1 = w1 * h1
    area2 = w2 * h2
    union_area = area1 + area2 - inter_area + eps

    # IoU dasar
    iou = inter_area / union_area

    # Lebar dan tinggi kotak terkecil yang membungkus box1 dan box2
    cw = torch.max(b1_x2, b2_x2) - torch.min(b1_x1, b2_x1)
    ch = torch.max(b1_y2, b2_y2) - torch.min(b1_y1, b2_y1)

    # Diagonal kotak pembungkus
    c2 = cw.pow(2) + ch.pow(2) + eps

    # titik tengah bbox prediksi
    b1_center_x = (b1_x1 + b1_x2) / 2
    b1_center_y = (b1_y1 + b1_y2) / 2
    
    # titik tengah bbox ground truth
    b2_center_x = (b2_x1 + b2_x2) / 2
    b2_center_y = (b2_y1 + b2_y2) / 2

    center_distance = (b2_center_x - b1_center_x).pow(2) + (b2_center_y - b1_center_y).pow(2) 
    
    # Penalti bentuk bbox
    v = (4 / math.pi**2) * ( torch.atan(w2 / h2) - torch.atan(w1 / h1)).pow(2)

    # bobot untuk penalti bentuk bbox
    with torch.no_grad():
        alpha = v / (1 - iou + v + eps) 

    # CIoU
    ciou = iou - (center_distance / c2 + alpha * v) # bisa memiliki nilai negatif

    return ciou # [N, 1]

In [ ]:
def xywh2xyxy(x):
    """
    Mengubah format bounding box dari xywh ke xyxy.

    xywh = [x_center, y_center, width, height]
    xyxy = [x1, y1, x2, y2]
    """
    y = x.clone()

    x_center = x[..., 0]
    y_center = x[..., 1]
    width    = x[..., 2]
    height   = x[..., 3]

    # koordinat sudut kiri atas
    y[..., 0] = x_center - width / 2  
    y[..., 1] = y_center - height / 2
    
    # koordinat sudut kanan bawah
    y[..., 2] = x_center + width / 2
    y[..., 3] = y_center + height / 2

    return y

def xyxy2xywh(x):
    """
    Mengubah bbox dari xyxy ke xywh.

    xyxy = [x1, y1, x2, y2]
    xywh = [x_center, y_center, width, height]
    """
    y = x.clone()

    x1 = x[..., 0]
    y1 = x[..., 1]
    x2 = x[..., 2]
    y2 = x[..., 3]
    
    y[..., 0] = (x1 + x2) / 2
    y[..., 1] = (y1 + y2) / 2
    y[..., 2] = x2 - x1
    y[..., 3] = y2 - y1

    return y

In [ ]:
class TaskAlignedAssigner(nn.Module):
    """
    menentukan anchor/grid point yang menjadi positive untuk tiap GT bbox
    
    B = batch size
    M = jumlah maksimum GT bbox per gambar dalam batch
    A = jumlah anchor/grid point total
    C = jumlah kelas deteksi (3)
    K = topk
    """
    def __init__(
        self,
        topk: int = 13,
        num_classes: int = 3,
        alpha: float = 0.5,
        beta: float = 6.0,
        stride: list = [8, 16, 32],
        eps: float = 1e-9,
    ):
        super().__init__()
        self.topk = topk # Jumlah Anchor Terbaik berdasarkan nilai alignment metric.
        self.num_classes = num_classes # Jumlah Class Detect 
        self.alpha = alpha 
        self.beta = beta
        self.stride = stride
        self.stride_val = self.stride[1] if len(self.stride) > 1 else self.stride[0] # memperbesar bbox kecil ketika proses assignment.
        self.eps = eps # menghindarai pembagian nol 

    def select_candidates_in_gts(self, xy_centers, gt_bboxes, mask_gt, eps=1e-9):
        """ 
        Mengecek anchor point mana yang berada di dalam tiap GT bbox.

        Input:
            xy_centers:
                Tensor shape [A, 2].
                Berisi koordinat pusat anchor point.

            gt_bboxes:
                Tensor shape [B, M, 4].
                Format bbox: [x1, y1, x2, y2].

            mask_gt:
                Tensor shape [B, M, 1].
                GT bbox yang valid.
                1/True = bbox asli.
                0/False = bbox padding.

        Output:
            bbox_deltas = Tensor boolean shape [B, M, A].
                True    = anchor berada di dalam GT bbox.
                False   = anchor berada di luar GT bbox.
        """
        
        gt_bboxes_xywh = xyxy2xywh(gt_bboxes) # [x1, y1, x2, y2] => [x_center, y_center, width, height]
        wh_mask = gt_bboxes_xywh[..., 2:] < self.stride[0]  # apakah width < 8 atau height < 8 (True / False)  [B, M, 2]
        
        # Untuk object kecil, area pencarian anchor diperbesar agar anchor yang masuk ke dalam bbox tidak terlalu sedikit
        gt_bboxes_xywh[..., 2:] = torch.where( 
            (wh_mask * mask_gt).bool(),
            torch.tensor(self.stride_val, dtype=gt_bboxes_xywh.dtype, device=gt_bboxes_xywh.device),
            gt_bboxes_xywh[..., 2:],
        )   
        gt_bboxes = xywh2xyxy(gt_bboxes_xywh) # [x_center, y_center, width, height] => [x1, y1, x2, y2]

        n_anchors = xy_centers.shape[0] # Jumah Anchor Point
        bs, n_boxes, _ = gt_bboxes.shape # battch size & maks object per gambar
        lt, rb = gt_bboxes.view(-1, 1, 4).chunk(2, 2)  # [x1,y1],[x2,y2]  shape = [B*M, 1, 2]
        
        # menghitung jarak anchor ke sisi bbox  
        # hasil concat = [LD, TD, RD, BD] ; hasil view = [B, M, A, 4]
        bbox_deltas = torch.cat((xy_centers[None] - lt, rb - xy_centers[None]), dim=2).view(bs, n_boxes, n_anchors, -1)
        
        # nilai terkecil dari 4 jarak, Kalau nilai terkecil masih lebih dari eps = semua jarak positif.
        bbox_deltas= bbox_deltas.amin(dim=3).gt_(eps) # [B, M, A, 4] 
        return bbox_deltas # [B, M, A]

    def iou_calculation(self, gt_bboxes, pd_bboxes):
        """ 
            Menghitung overlap antara GT bbox dan predicted bbox menggunakan CIoU.

            Input:
                gt_bboxes: 
                    Tensor shape [N, 4].
                    Format [x1, y1, x2, y2].

                pd_bboxes:
                    Tensor shape [N, 4].
                    Format [x1, y1, x2, y2].

            Output:
                Tensor shape [N].
                nilai CIoU yang di-clamp minimal 0.
        """
        return bbox_ciou(gt_bboxes, pd_bboxes).squeeze(-1).clamp_(0) # [N,1] => [N] positif or 0

    def get_box_metrics(self, pd_scores, pd_bboxes, gt_labels, gt_bboxes, mask_gt):
        """
        Menghitung alignment metric untuk tiap pasangan GT bbox dan anchor.

        Input:
            pd_scores: Skor/probabilitas class prediksi untuk tiap anchor.
                Tensor shape [B, A, C].

            pd_bboxes: Predicted bbox untuk tiap anchor.
                Tensor shape [B, A, 4].
                Format [x1, y1, x2, y2].

            gt_labels: Label class untuk tiap GT bbox.
                Tensor shape [B, M, 1].

            gt_bboxes:
                Tensor shape [B, M, 4].
                Format [x1, y1, x2, y2].

            mask_gt: pasangan GT-anchor yang valid.
                Tensor boolean shape [B, M, A].

        Output:
            align_metric: Nilai kualitas anchor berdasarkan class score dan overlap.
                Tensor shape [B, M, A].

            overlaps: Nilai CIoU antara GT bbox dan predicted bbox.
                Tensor shape [B, M, A].
        """
        
        na = pd_bboxes.shape[-2] # A 
        mask_gt = mask_gt.bool()  # [B, M, A]
        
        # tensor kosong berisi overlap antara predicted bbox dan GT bbox [B, M, A]
        overlaps = torch.zeros([self.bs, self.n_max_boxes, na], dtype=pd_bboxes.dtype, device=pd_bboxes.device)
        
        # tensor kosong berisi skor kelas prediksi untuk kelas GT [B, M, A]
        bbox_scores = torch.zeros([self.bs, self.n_max_boxes, na], dtype=pd_scores.dtype, device=pd_scores.device) 
        
        # tensor kosong berisi [2, B, M]    
        ind = torch.zeros([2, self.bs, self.n_max_boxes], dtype=torch.long)  
        ind[0] = torch.arange(end=self.bs).view(-1, 1).expand(-1, self.n_max_boxes)  # [B,M] Batch Index
        ind[1] = gt_labels.squeeze(-1)  # [B, M, 1] => [B, M] Class GT
        
        # mengambil score prediksi yang sesuai dengan class GT
        # pd_scores shape: [B, A, C]
        # hasil indexing: [B, M, A]
        # hanya posisi valid (mask_gt=True) yang disalin ke bbox_scores
        bbox_scores[mask_gt] = pd_scores[ind[0], :, ind[1]][mask_gt]  # [B, M, A]

        # [B, A, 4] => [B, 1, A, 4] => [B, M, A, 4] => [N, 4]  N = Jumah elemen True pada mask_gt
        pd_boxes = pd_bboxes.unsqueeze(1).expand(-1, self.n_max_boxes, -1, -1)[mask_gt]
        
        # [B, M, 4] => [B, M, 1, 4] => [B, M, A, 4] => [N, 4] N = Jumah eleme True pada mask_gt
        gt_boxes = gt_bboxes.unsqueeze(2).expand(-1, -1, na, -1)[mask_gt]
        
        # [B, M, A] => [N, 4]
        overlaps[mask_gt] = self.iou_calculation(gt_boxes, pd_boxes) 
        # [B, M, A]
        align_metric = bbox_scores.pow(self.alpha) * overlaps.pow(self.beta) # align_metric = bbox_scores^0.5 X overlaps^6
        return align_metric, overlaps 

    def select_topk_candidates(self, metrics, topk_mask=None):
        """
        Memilih top-k anchor terbaik untuk tiap GT bbox berdasarkan alignment metric.

        Input:
            metrics:
                Tensor shape [B, M, A].
                
            topk_mask: 
                hasil top-k valid atau tidak. untuk mengabaikan GT bbox padding.(optional)
                tensor shape [B, M, K].
                
        Output:
            Tensor shape [B, M, A].
            Berisi mask top-k:
                1 = anchor termasuk top-k untuk GT tersebut.
                0 = anchor tidak termasuk top-k.
        """
        
        # ambil K elemen terbesar dari dim =-1 tensor metrics  
        # [B, M, A] => [B, M, K] (Value), [B, M, K](Indeks)
        topk_metrics, topk_idxs = torch.topk(metrics, self.topk, dim=-1, largest=True)
        
        # jika seluruh metric bernilai nol = GT dianggap tidak valid
        if topk_mask is None: # [B, M, K] => [B, M, 1] => [B, M, K] Boolean 
            # cek GT memiliki minimal satu metric > eps
            # # [B,M,K] -> [B,M,1] -> [B,M,K] Boolean   
            topk_mask = (topk_metrics.max(-1, keepdim=True)[0] > self.eps).expand_as(topk_idxs)
            
        # [B, M, K] nilai False = 0
        topk_idxs.masked_fill_(~topk_mask, 0) # top-k yang tidak valid diganti index 0

        # [B, M, A]  mask output (niai 0)
        count_tensor = torch.zeros(metrics.shape, dtype=torch.int8, device=topk_idxs.device)
        
        # [B, M, 1] untuk nilai akan ditambahkan oleh scatter_add_
        ones = torch.ones_like(topk_idxs[:, :, :1], dtype=torch.int8, device=topk_idxs.device)
        
        for k in range(self.topk): # 0-K-1
            # Perluas topk_idxs untuk setiap nilai k dan tambahkan 1 pada posisi yang ditentukan
            count_tensor.scatter_add_(-1, topk_idxs[:, :, k : k + 1], ones) # daftar index top-k => mask biner [B,M,A]

        count_tensor.masked_fill_(count_tensor > 1, 0) # anchor terpilih lebih dari sekali = tidak valid 

        return count_tensor.to(metrics.dtype) 

    def get_pos_mask(self, pd_scores, pd_bboxes, gt_labels, gt_bboxes, anc_points, mask_gt):
        """
        Membuat mask positive candidate untuk tiap GT bbox.

        Input:
            pd_scores: Skor class prediksi untuk tiap anchor.
                Tensor shape [B, A, C].
                
            pd_bboxes: Predicted bbox untuk tiap anchor.
                Tensor shape [B, A, 4].
                Format [x1, y1, x2, y2].

            gt_labels:
                Tensor shape [B, M, 1].

            gt_bboxes:
                Tensor shape [B, M, 4].
                Format [x1, y1, x2, y2].

            anc_points: Koordinat pusat anchor/grid point.
                Tensor shape [A, 2].

            mask_gt: GT bbox valid dan padding.
                Tensor shape [B, M, 1].
                
        Output: Mask positive candidate.
            mask_pos:
                Tensor shape [B, M, A].

            align_metric: Nilai alignment metric.
                Tensor shape [B, M, A].

            overlaps: Nilai CIoU antara GT dan prediksi.
                Tensor shape [B, M, A].
        """
        # Anchor in GT BBox [B, M, A] 
        mask_in_gts = self.select_candidates_in_gts(anc_points, gt_bboxes, mask_gt)
        
        # [B, M, A], [B, M, A]
        align_metric, overlaps = self.get_box_metrics(pd_scores, pd_bboxes, gt_labels, gt_bboxes, mask_in_gts * mask_gt)
        
        # [B, M, A]
        mask_topk = self.select_topk_candidates(align_metric, topk_mask=mask_gt.expand(-1, -1, self.topk).bool())
        
        # [B, M, A] * [B, M, A] * [B, M, 1]
        mask_pos = mask_topk * mask_in_gts * mask_gt # mask_pos = mask_topk AND mask_in_gts AND mask_gt == 1

        return mask_pos, align_metric, overlaps

    def select_highest_overlaps(self, mask_pos, overlaps, n_max_boxes):
        """
        Menentukan GT bbox final untuk tiap anchor positive.

        Input:
            mask_pos : Mask positive candidate untuk tiap pasangan GT-anchor
                Tensor shape [B, M, A].

            overlaps : Nilai CIoU antara GT bbox dan predicted bbox.
                Tensor shape [B, M, A].

            n_max_boxes : Jumlah maksimum GT bbox dalam satu gambar, M.
                Int

        Output: Index GT bbox yang ditugaskan untuk tiap anchor
            target_gt_idx :
                Tensor shape [B, A].

            fg_mask :
                Tensor shape [B, A].
                1/True jika anchor adalah foreground/positive.
                0/False jika anchor adalah background.

            mask_pos : Mask final setelah konflik antar-GT diselesaikan.
                Tensor shape [B, M, A].
        """
        
        # [B, M, A] => [B, A] boolean
        fg_mask = mask_pos.sum(-2)
        if fg_mask.max() > 1:  # cek apakah 1 anchor dipilih oleh beberapa GT 
            # [B, A] => [B, 1, A] => [B, M, A]
            mask_multi_gts = (fg_mask.unsqueeze(1) > 1).expand(-1, n_max_boxes, -1) # cek konnflik, fg_mask >1 = True

            # [B, M, A] => [B, A] cari index yang memiliki nilai terbesar 
            max_overlaps_idx = overlaps.argmax(dim=1)  # tiap anchor, pilih GT dengan IoU terbesar
            
            # [B, M, A]
            is_max_overlaps = torch.zeros(mask_pos.shape, dtype=mask_pos.dtype, device=mask_pos.device)
            is_max_overlaps.scatter_(1, max_overlaps_idx.unsqueeze(1), 1)
            
            # if mask_multi_gts True (Anchor konflik) → pakai GT dengan IoU terbesar, else → biarkan seperti semula
            mask_pos = torch.where(mask_multi_gts, is_max_overlaps, mask_pos).float()  # [B, M, A]

            # [B, M, A] => [B, A]
            fg_mask = mask_pos.sum(-2) # 1= foreground; 0=background

        # Tiap anchor, di arahkan ke GT mana 
        target_gt_idx = mask_pos.argmax(-2)  # [B, A]
        return target_gt_idx, fg_mask, mask_pos # [B, A], [B, A], [B, M, A]

    def get_targets(self, gt_labels, gt_bboxes, target_gt_idx, fg_mask):
        """
        Membuat target label, target bbox, dan target score untuk tiap anchor.
        (mengubah target_gt_idx jadi  target_labels, target_bboxes dan target_scores)
        
        Input:
            gt_labels: Label class untuk tiap GT bbox
                Tensor shape [B, M, 1].

            gt_bboxes:
                Tensor shape [B, M, 4].
                Format [x1, y1, x2, y2].

            target_gt_idx: Index GT bbox yang ditugaskan untuk tiap anchor.
                Tensor shape [B, A].

            fg_mask: anchor foreground/positive.
                Tensor shape [B, A].

        Output:
            target_labels: Label class target untuk tiap anchor.
                Tensor shape [B, A].
                
            target_bboxes: Target bbox untuk tiap anchor.
                Tensor shape [B, A, 4].

            target_scores: One-hot target class score.
                Tensor shape [B, A, C].
                Anchor background memiliki nilai [0, 0, 0].
        """
        
        # torch.arrange(B) => [B] => [B, 1]
        batch_ind = torch.arange(end=self.bs, dtype=torch.int64, device=gt_labels.device)[..., None]
        
        # [B, A] = [B, A] + [B, 1] * M
        target_gt_idx = target_gt_idx + batch_ind * self.n_max_boxes 
        
        # [B, M, 1] -> flatten() -> [B*M]
        # indexing dengan target_gt_idx [B, A]
        # hasil -> [B, A]
        target_labels = gt_labels.long().flatten()[target_gt_idx]  # Label GT untuk setiap anchor.

        # [B, M, 4] -> [B*M, 4]
        # indexing dengan target_gt_idx [B,A]
        # hasil -> [B,A,4]
        target_bboxes = gt_bboxes.view(-1, gt_bboxes.shape[-1])[target_gt_idx] # BBox GT yang ditugaskan ke setiap anchor.

        # [B, A]
        target_labels.clamp_(0) # (padding/background menjadi minimal 0)

        # [B, A, C]
        target_scores = torch.zeros(
            (target_labels.shape[0], target_labels.shape[1], self.num_classes),
            dtype=torch.int64,
            device=target_labels.device,
        )  
        # target_labels [B,A] -> unsqueeze(-1) [B,A,1]
        # scatter membuat one-hot class target
        # output  [B,A,C]
        target_scores.scatter_(2, target_labels.unsqueeze(-1), 1)

        # [B, A] => [B, A, 1] => [B, A, C]
        fg_scores_mask = fg_mask[:, :, None].repeat(1, 1, self.num_classes) 
        
        # Background anchor -> semua class = 0
        # Foreground anchor -> one-hot target
        target_scores = torch.where(fg_scores_mask > 0, target_scores, 0) # One-hot class target utnuk CLS Loss (BCEWithLogitLoss).

        return target_labels, target_bboxes, target_scores

    @torch.no_grad()
    def forward(self, pd_scores, pd_bboxes, anc_points, gt_labels, gt_bboxes, mask_gt):
        """
        Proses utama task-aligned assignment ketika terdapat GT bbox.

        - get_pos_mask ()
            ->  select_candidates_in_gts () 
            ->  select_topk_candidates ()
            ->  get_box_metrics ()
                -->	iou_calculation ()
        - select_highest_overlaps ()
        - get_target ()

        Langkah:
            1. Cari anchor positive candidate.
            2. Selesaikan konflik jika satu anchor dipilih oleh beberapa GT.
            3. Buat target label, bbox, dan score.
            4. Normalisasi target score berdasarkan alignment metric dan overlap.

        Input:
            pd_scores: probabilitas 
                Tensor shape [B, A, C].

            pd_bboxes:
                Tensor shape [B, A, 4].

            anc_points:
                Tensor shape [A, 2].

            gt_labels:
                Tensor shape [B, M, 1].

            gt_bboxes:
                Tensor shape [B, M, 4].

            mask_gt:
                Tensor shape [B, M, 1].

        Output:
            target_labels:
                Tensor shape [B, A].

            target_bboxes: masih dalam bentuk pixel
                Tensor shape [B, A, 4].

            target_scores: Soft target score setelah dinormalisasi.
                Tensor shape [B, A, C].

            fg_mask: Mask foreground anchor 
                Tensor boolean shape [B, A].
                True    = anchor positive.
                False   = background anchor
 
            target_gt_idx: Index GT yang ditugaskan ke setiap anchor.
                Tensor shape [B, A].
        """
        self.bs = pd_scores.shape[0]
        self.n_max_boxes = gt_bboxes.shape[1]
        
        # [B, M, A], [B, M, A], [B, M, A]
        mask_pos, align_metric, overlaps = self.get_pos_mask(
            pd_scores, pd_bboxes, gt_labels, gt_bboxes, anc_points, mask_gt
        )

        # [B, A], [B, A], [B, M, A]
        target_gt_idx, fg_mask, mask_pos = self.select_highest_overlaps(
            mask_pos, overlaps, self.n_max_boxes
        )

        # [B, A], [B, A, 4], [B, A, C]
        target_labels, target_bboxes, target_scores = self.get_targets(gt_labels, gt_bboxes, target_gt_idx, fg_mask)

        # Normalize
        align_metric *= mask_pos # [B, M, A] Hapus Anchor yang bukan positive (Mempertahankan aligment metric)
        pos_align_metrics = align_metric.amax(dim=-1, keepdim=True)  # cari aligment metric terbesar diantara positive anchor [B, M, 1]
        pos_overlaps = (overlaps * mask_pos).amax(dim=-1, keepdim=True)  # cari IOU terbesar diantara positive anchor [B, M, 1]
        norm_align_metric = (align_metric * pos_overlaps / (pos_align_metrics + self.eps)) # Normalisasi alignment metric ke skala IoU [B, M, A]
        norm_align_metric = norm_align_metric.amax(-2).unsqueeze(-1) # ambil nilai normalisasi terbesar [B, M, A] => [B, A] => [B, A, 1]
        target_scores = target_scores * norm_align_metric # mengubah one hot menjadi soft label

        return target_labels, target_bboxes, target_scores, fg_mask.bool(), target_gt_idx

In [ ]:
def make_anchors(feats, strides, grid_cell_offset=0.5): 
    """
    feats[0].shape = [B, 67, 80, 80]   # stride 8
    feats[1].shape = [B, 67, 40, 40]   # stride 16
    feats[2].shape = [B, 67, 20, 20]   # stride 32
    
    To 
    
    anchor_points.shape = [8400, 2] (8400 titik anchor & tiap titik memiliki x_anchor, y_anchor)
    8400 = 80*80 + 40*40 + 20*20
    2    = koordinat [x, y]

    stride_tensor.shape = [8400, 1]
    berisi stride untuk setiap anchor point
    
    anchor_points = posisi titik anchor/grid (acuan untuk membuat bbox dari prediksi jarak)
    stride_tensor = skala untuk mengubah koordinat feature map ke koordinat gambar asli
    """
    anchor_points, stride_tensor = [], []
    assert feats is not None
    dtype, device = feats[0].dtype, feats[0].device
    
    # Loop tiap feature map YOLO
    for i in range(len(feats)): 
        stride = strides[i] # 8, 16, 32
        h, w = feats[i].shape[2:]  # [80,80], [40, 40], [20, 20]  => [80, 40, 20], [80, 40, 20]
        sx = torch.arange(end=w, device=device, dtype=dtype) + grid_cell_offset  # [0.5, 1.5, 2.5 ... len(w[feats])]
        sy = torch.arange(end=h, device=device, dtype=dtype) + grid_cell_offset  # [0.5, 1.5, 2.5 ... len(h[feats])]
        sy, sx = torch.meshgrid(sy, sx, indexing="ij") # 1D => 2D
        
        anchor_points.append(torch.stack((sx, sy), dim=-1).view(-1, 2)) # 2D -> 3D :  [80, 80, 2] => [6400, 2] 
        stride_tensor.append(torch.full((h * w, 1), stride, dtype=dtype, device=device)) # isi nilai stride_tensor jadi stride sebanyak w*x 
        
    anchor_points = torch.cat(anchor_points, dim=0) # [8400, 2]
    stride_tensor = torch.cat(stride_tensor, dim=0) # [8400, 1]
        
    return anchor_points, stride_tensor

def dist2bbox(distance, anchor_points, xywh=True, dim=-1):
    """
    Mengubah jarak dari anchor point menjadi bounding box.
    distance.shape [B, 8400, 4]
    [left, top, right, bottom] HANYA JARAK DARI ANCHOR POINT
    
    To 
    
    xywh=True:  [x_center, y_center, width, height]
    
    xywh=False: [x1, y1, x2, y2]   KOORDINAT BBOX
    """
    lt, rb = distance.chunk(2, dim) 
    x1y1 = anchor_points - lt 
    x2y2 = anchor_points + rb 
    if xywh:
        c_xy = (x1y1 + x2y2) / 2 # Center Point (titik tengah dari bbox hasil prediksi)
        wh = x2y2 - x1y1 
        return torch.cat([c_xy, wh], dim)  # xywh bbox
    return torch.cat((x1y1, x2y2), dim)  # xyxy bbox

def bbox2dist(anchor_points: torch.Tensor, bbox: torch.Tensor, reg_max: int | None = None) -> torch.Tensor:
    """
    membuat target jarak dari ground truth bbox terhadap anchor point
    
    anchor_points:
        Format:
            [x_anchor, y_anchor]

    bbox:
        Format:
            [x1, y1, x2, y2]

    reg_max:
        Batas maksimum jarak untuk DFL loss.
        Jika reg_max=16, maka jarak dibatasi ke rentang:
            0 sampai 15.99
    
    TO 
    
    distance (Jarak dari anchor point ke sisi bbox)
        Format:
            [left, top, right, bottom]
    """
    x1y1, x2y2 = bbox.chunk(2, -1)  # [x1, y1, x2, y2] -> x1y1, 21y2 = [x1, y1], [x2, y2]
    dist = torch.cat((anchor_points - x1y1, x2y2 - anchor_points), dim=-1) 
    if reg_max is not None:
        dist = dist.clamp_(0, reg_max - 0.01)  # dist (lt, rb)
    return dist

In [ ]:
class DFLoss(nn.Module):
    def __init__(self, reg_max: int = 16) -> None:
        super().__init__()
        self.reg_max = reg_max
    
    def forward(self, pred_dist: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """
        pred_dist:
            Prediksi distribusi.
            Shape umum: [N, reg_max]
            Format: [jumlah_fg * 4, 16]

        target:
            Target jarak dari anchor point ke bbox.
            Shape umum: [jumlah_fg, 4]
            Format: [left, top, right, bottom]
            
        TO 
        
        shape = [jumlah_fg, 1]
        """
        
        target = target.clamp_(0, self.reg_max - 1 - 0.01) # supaya tr = tl + 1 tidak menjadi indeks 16
        tl = target.long() #target lower / target kiri pada distribusi
        tr = tl + 1 #target upper / target kanan pada distribusi
        wl = tr - target  # weight left
        wr = 1 - wl  # weight right
        
        return (
            F.cross_entropy(pred_dist, tl.view(-1), reduction="none").view(tl.shape) * wl
            + F.cross_entropy(pred_dist, tr.view(-1), reduction="none").view(tl.shape) * wr
        ).mean(-1, keepdim=True)

In [ ]:
class BboxLoss(nn.Module):
    """ 
    Menghitung bbox loss YOLO berbasis CIoU + DFL.

    Input:
        pred_dist:
            Prediksi distribusi bbox dari model.
            Shape: [B, A, 4 * reg_max]

        pred_bboxes:
            Bbox hasil decode dari model.
            Shape: [B, A, 4]
            Format: [x1, y1, x2, y2]

        anchor_points:
            Shape: [A, 2]

        target_bboxes:
            Bbox target hasil assigner.
            Shape: [B, A, 4]
            Format: [x1, y1, x2, y2]

        target_scores:
            Skor target/class hasil assigner.
            Shape: [B, A, nc]

        target_scores_sum:
            Jumlah total target score.
            Bentuk: scalar

        fg_mask:
            Mask anchor positif.
            Shape: [B, A]

    Output:
        loss_iou:
            Loss posisi bbox berdasarkan CIoU.

        loss_dfl:
            Loss distribusi jarak bbox berdasarkan DFL.
    """ 
    
    def __init__(self, reg_max: int = 16):
        super().__init__()
        self.dfl_loss = DFLoss(reg_max) 

    def forward(
        self,
        pred_dist: torch.Tensor, # [B, A, 4 * reg_max]
        pred_bboxes: torch.Tensor, # [B, A, 4]
        anchor_points: torch.Tensor, # [A, 2]
        target_bboxes: torch.Tensor, # [B, A, 4]
        target_scores: torch.Tensor, # [B, A, nc]
        target_scores_sum: torch.Tensor, # scalar
        fg_mask: torch.Tensor, # [B, A] mask untuk memilih anchor yang dianggap foreground/positif
    ) -> tuple[torch.Tensor, torch.Tensor]:

        # [B, A, nc] -> [B, A] -> [jumlah_fg] -> [jumlah_fg, 1]
        weight = target_scores.sum(-1)[fg_mask].unsqueeze(-1) 
        iou = bbox_ciou(pred_bboxes[fg_mask], target_bboxes[fg_mask]) 
        loss_iou = ((1.0 - iou) * weight).sum() / target_scores_sum #dikalil weight dan di normalisasi 

        # [x1, y1, x2, y2] => [left, top, right, bottom] dari anchor point ke bbox target.
        target_ltrb = bbox2dist(anchor_points, target_bboxes, self.dfl_loss.reg_max - 1) # kelas distribusi dfl 0-15
        
        # [jumlah_fg, 4 * reg_max] => [jumlah_fg, 64] =>  [jumlah_fg * 4, 16]
        pred_dist_fg = pred_dist[fg_mask].view(-1, self.dfl_loss.reg_max)
        
        # dfl_loss ([jumlah_fg * 4, 16], [jumlah_fg , 4]) => [jumlah_fg, 1]
        loss_dfl = self.dfl_loss (pred_dist_fg, target_ltrb[fg_mask])
        loss_dfl = loss_dfl.sum() / target_scores_sum  # Normalisasi Loss

        return loss_iou, loss_dfl

In [ ]:
class v8DetectionLoss(nn.Module):
    """ 
    
    B = batch size
    M = jumlah maksimum GT bbox per gambar dalam batch
    N = total seluruh GT bbox dalam batch.
    """
    def __init__(self, model: torch.nn.Module, tal_topk: int = 10):
        super().__init__()
        
        device = next(model.parameters()).device

        # Bobot Hyperparameter loss YOLO    
        h = SimpleNamespace(
            box=7.5,
            cls=0.5,
            dfl=1.5
        )

        self.bce = nn.BCEWithLogitsLoss(reduction="none") # Classification Loss
        self.hyp = h # Hyperparameter loss

        # Ambil atribut dari model custom
        self.stride = model.stride.to(device) if isinstance(model.stride, torch.Tensor) else torch.tensor(model.stride, device=device)
        self.nc = model.nc # 3
        self.reg_max = model.reg_max # jumlah bin distribus DFL
        self.no = self.nc + self.reg_max * 4 # total chanel output pada head (3 + 16*4) = 67
        self.device = device

        # Menentukan :  anchor positif & negatif, target bbox & class    
        
        self.assigner = TaskAlignedAssigner(
            topk=tal_topk, # 10
            num_classes=self.nc, # 3
            alpha=0.5,
            beta=6.0,
            stride=self.stride.tolist() 
        )

        self.bbox_loss = BboxLoss(self.reg_max).to(device) # IOU & DFL LOSS
        
        # mengubah distribusi probabilitas menjadi jarak kontinu.
        self.proj = torch.arange(self.reg_max, dtype=torch.float, device=device) # [0-15]

    def preprocess(self, targets: torch.Tensor, batch_size: int, scale_tensor: torch.Tensor) -> torch.Tensor:
        """
        Mengubah target anotasi dari format gabungan seluruh batch menjadi format per gambar.
        Mengubah bbox dari xywh ke xyxy.

        Input:

            targets:
                Tensor shape [N, 6]
                Format: [batch_idx, class, x, y, w, h]

            scale_tensor: 
                Tensor shape [4]

                Digunakan untuk mengubah bbox
                dari koordinat normalized
                menjadi koordinat pixel.

                Format: [img_w, img_h, img_w, img_h]

        Output:
            out:
                Tensor shape [B, M, 5]
                Format: [class, x1, y1, x2, y2]
        """
        # jumlah GT bbox seluruh batch, jumlah elemen per target
        nl, ne = targets.shape # N, 6
        batch_idx = targets[:, 0].long()  # [N]
        _, counts = batch_idx.unique(return_counts=True) # Jumlah GT Tiap Gambar[N]
        counts = counts.to(dtype=torch.int32)
        
        out = torch.zeros(batch_size, counts.max(), ne - 1, device=self.device) # [B, M, 5] batch pindah
        
        offsets = torch.zeros(batch_size + 1, dtype=torch.long, device=self.device) # menyediakan satu slot tambahan di depan untuk angka 0 
        # Hitung Banyak GT per Gambar
        offsets.scatter_add_(0, batch_idx + 1, torch.ones_like(batch_idx)) # menggeser index supaya slot 0 tetap kosong
        offsets = offsets.cumsum(0) # jumlah gt per gambar => posisi awal GT tiap gambar  [B+1]
        
        #Index Dalam Gambar (nilai within_idx tidak bisa sama dengan batch idx)
        within_idx = torch.arange(nl, device=self.device) - offsets[batch_idx] # menghasilkan nomor urut GT di dalam gambar.[N] 
        
        out[batch_idx, within_idx] = targets[:, 1:] # [B, M, 5]
        # ambil 4 kolom belakang untuk di ubah jadi xyxy
        out[..., 1:5] = xywh2xyxy(out[..., 1:5].mul_(scale_tensor)) # ubah format dari
        
        return out

    def bbox_decode(self, anchor_points: torch.Tensor, pred_dist: torch.Tensor) -> torch.Tensor:
        """
        Mengubah output distribusi DFL menjadi bounding box.

        Input:
            anchor_points:
                Tensor shape [A, 2].
                Format: [x_anchor, y_anchor]

            pred_dist:
                Tensor shape [B, A, 4*reg_max].

        Output:
            pred_bboxes:
                Tensor shape [B, A, 4].
                Format:[x1, y1, x2, y2]
        """
        b, a, c = pred_dist.shape  # [B, A, 64]
        
        #[B, A, 64] => [B, A, 4, 16] logits => [B, A, 4, 16] prob 
        pred_dist = pred_dist.view(b, a, 4, c // 4).softmax(3)
        
        # [B, A, 4, 16] distribusi => [B, A, 4]nilai jarak menggunakan expected value 
        pred_dist = pred_dist.matmul(self.proj.type(pred_dist.dtype))
        
        pred_bboxes = dist2bbox(pred_dist, anchor_points, xywh=False)
        return pred_bboxes

    def get_assigned_targets_and_loss(self, preds: dict[str, torch.Tensor], batch: dict[str, Any]) -> tuple:
        """
        Melakukan proses assignment target dan menghitung loss YOLO.

        Langkah:
            1. Mengubah output detection head menjadi format yang siap digunakan.
            2. Membuat anchor points.
            3. Menyiapkan GT bbox dan GT label.
            4. Decode output DFL menjadi bbox prediksi.
            5. Menjalankan TaskAlignedAssigner.
            6. Menghitung Classification Loss (BCE).
            7. Menghitung Box Loss (IoU).
            8. Menghitung DFL Loss.

        Input:
            preds: Dictionary hasil parse_output().
                boxes:
                    Tensor [B, 4*reg_max, A]

                scores:
                    Tensor [B, nc, A]

                feats:
                    List feature map detection head

            batch: Ground truth satu batch.
                batch_idx:
                    Tensor [N]

                cls:
                    Tensor [N]

                bboxes:
                    Tensor [N,4]
                    format xywh normalized

        Output:
            loss:
                Tensor [3] : [box_loss, cls_loss, dfl_loss]

            loss_detach: Versi tanpa gradient untuk logging.
                Tensor [3]
        """
        loss = torch.zeros(3, device=self.device)  # box, cls, dfl [3]
        pred_distri, pred_scores = (
            preds["boxes"].permute(0, 2, 1).contiguous(), # [B, A, 4*reg_max]
            preds["scores"].permute(0, 2, 1).contiguous(), # [B, A, C] logit 
        )
        anchor_points, stride_tensor = make_anchors(preds["feats"], self.stride, 0.5) # [A, 2], [A, 1]

        dtype = pred_scores.dtype
        batch_size = pred_scores.shape[0]  
        
        # [640, 640]
        imgsz = torch.tensor(preds["feats"][0].shape[2:], device=self.device, dtype=dtype) * self.stride[0]

        # ([N,1], [N, 1], [N, 4])dim=1 => [N, 6] (img_idx, class, x, y, w, h)
        targets = torch.cat(
            (batch["batch_idx"].view(-1, 1), batch["cls"].view(-1, 1), batch["bboxes"])
            , dim=1)
        
        #[N, 6]  => [B, M, 5]
        targets = self.preprocess(targets.to(self.device), batch_size, scale_tensor=imgsz[[1, 0, 1, 0]])
        
        gt_labels, gt_bboxes = targets.split((1, 4), 2)  # cls [B, M, 1], xyxy [B, M, 4]
        mask_gt = gt_bboxes.sum(2, keepdim=True).gt_(0.0) # [B, M, 1] 1= BBox asli; 0=Padding

        # prediksi jarak dari model + anchor point → bbox prediksi 
        pred_bboxes = self.bbox_decode(anchor_points, pred_distri)  # [B, A, 4] masih daam koordinat

        _, target_bboxes, target_scores, fg_mask, target_gt_idx = self.assigner(
            pred_scores.detach().sigmoid(), # diubah menjadi probabilitas 
            (pred_bboxes.detach() * stride_tensor).type(gt_bboxes.dtype), # ubah jadi pixel image
            anchor_points * stride_tensor, # di kali karna output pada yolo itu feature map koordinat, bukan input img
            gt_labels,
            gt_bboxes,
            mask_gt,
        )

        target_scores_sum = max(target_scores.sum(), 1) # Jumlah Score Positif

        # Cls loss with optional class weighting
        bce_loss = self.bce(pred_scores, target_scores.to(dtype))  # [B, A, C]
        loss[1] = bce_loss.sum() / target_scores_sum  # BCE

        # Bbox loss
        if fg_mask.sum():
            loss[0], loss[2] = self.bbox_loss(
                pred_distri,
                pred_bboxes,
                anchor_points,
                target_bboxes / stride_tensor,
                target_scores,
                target_scores_sum,
                fg_mask
            )
        loss[0] *= self.hyp.box  # box gain
        loss[1] *= self.hyp.cls  # cls gain
        loss[2] *= self.hyp.dfl  # dfl gain
        
        return loss, loss.detach() # loss(box, cls, dfl)
         
    
    def loss(self, preds: dict[str, torch.Tensor], batch: dict[str, torch.Tensor]) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Menghitung detection loss untuk satu batch.

        Input:
            preds: Dictionary hasil parse_output().
                boxes:
                    Tensor shape [B, 4*reg_max, A].

                scores:
                    Tensor shape [B, nc, A].

                feats:
                    List feature map detection head.

            batch: Ground truth satu batch.
                batch_idx:
                    Tensor shape [N].

                cls:
                    Tensor shape [N].

                bboxes: (normalized)
                    Tensor shape [N, 4].
                    Format: [x_center, y_center, width, height]  

        Output:
            loss: box_loss, cls_loss, dfl_loss (sudah dikali batch size)
                Tensor shape [3].

            loss_detach: untuk logging tanpa menyimpan computational graph.              
                Tensor shape [3].
        """
        batch_size = preds["boxes"].shape[0]
        loss, loss_detach = self.get_assigned_targets_and_loss(preds, batch) # assignment target & komponen loss
        return loss, loss_detach
    
    def parse_output(self, preds): 
        """
        Mengubah output detection head menjadi format dictionary untuk proses loss.
        
        Input:
            preds: List feature map dari detection head. (LIST !)

        Output:
            Dictionary:
                boxes: Prediksi regresi bbox sebelum proses decode DFL.
                    Tensor shape [B, 4*reg_max, A].

                scores: Logits klasifikasi.
                    Tensor shape [B, nc, A].

                feats: List feature map dari detection head.

        """
        feats = preds # List Feature Map 
        bs = feats[0].shape[0] # batch size
    
        pred_distri, pred_scores = torch.cat(
             # [B,no,H,W] -> [B,no,H*W]
            [xi.view(bs, self.no, -1) 
             for xi in feats
             ],
            dim=2 # Gabung semua anchor dari p3, p4, p5
        ).split(
            (
                self.reg_max * 4, # distribusi DFL bbox
                self.nc     # class logits
            ), 
            dim=1
        )
    
        return {
            "boxes": pred_distri, # prediksi regresi bbox sebelum decode. [B, 4*reg_max, A]
            "scores": pred_scores, # logit class  [B, nc, A]
            "feats": feats # List feature map asli.
        }

    def forward(
        self,
        preds: dict[str, torch.Tensor] | tuple[torch.Tensor, dict[str, torch.Tensor]],
        batch: dict[str, torch.Tensor],
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        1. Mengubah output detection head menjadi format dictionary.
        2. Menghitung Box Loss, Classification Loss, dan DFL Loss.

        Input:
            preds: Output detection head model. (Feature Map)
            
            batch:Ground truth satu batch.
                batch_idx:
                    Tensor shape [N].

                cls:
                    Tensor shape [N].

                bboxes:
                    Tensor shape [N,4].

        Output:
            loss: Loss sudah dikalikan batch size.
            
            loss_detach: Digunakan untuk logging.
                Tensor shape [3].
        """
        return self.loss(self.parse_output(preds), batch)

In [ ]:
def build_targets(targets, device):
    """
    Mengubah target dari format list per gambar menjadi format batch gabungan.
    menggabungkan label dari banyak gambar dalam satu batch
    Input:
        targets:
            List berisi target untuk setiap gambar dalam batch.
            Panjang list = batch size.

            Setiap target punya shape [N, 5],
            Format = [class, x_center, y_center, width, height]

    Output:
        Dictionary:
            batch_idx:
                Shape [total_object]
                Menunjukkan bbox berasal dari gambar batch ke berapa.

            cls:
                Shape [total_object]
                Class label setiap bbox.

            bboxes:
                Shape [total_object, 4]
                Koordinat bbox.
    """
    
    batch_idx_list = []
    cls_list = []
    bboxes_list = []

    for image_idx, target in enumerate(targets):
        target = target.to(device)
        
        # jumlah object pada gambar ke-image_idx
        batch_idx = torch.full( 
            (target.shape[0],), # panjang tensor (total objek dalam gambar ke image_idx)
            image_idx, # isi tensor dengan indeks gambar
            dtype=torch.long,
            device=device
        )

        cls = target[:, 0].long() # class object
        bboxes = target[:, 1:5].float() # [x_center, y_center, width, height]

        batch_idx_list.append(batch_idx) 
        cls_list.append(cls)
        bboxes_list.append(bboxes)

    return {
        "batch_idx": torch.cat(batch_idx_list, dim=0), 
        "cls": torch.cat(cls_list, dim=0),
        "bboxes": torch.cat(bboxes_list, dim=0),
    }

In [ ]:
class MultiTaskLoss(nn.Module):
    """
    Menghitung total loss.
    1. YOLO Detection Loss
        - Box Loss
        - Classification Loss
        - DFL Loss

    2. Temperature Classification Loss
        - CrossEntropyLoss

    Total Loss:
        total_loss = yolo_total_loss + lambda_temp * temp_loss
    """
    
    def __init__(self, model, lambda_temp=1.0, temp_class_weights=None):
        super().__init__()  
        self.yolo_loss = v8DetectionLoss(model) # Detection loss YOLO (Bbox, Cls, DFL)
        
        device = next(model.parameters()).device

        if temp_class_weights is not None:
            temp_class_weights = temp_class_weights.to(device)

        self.temp_loss_fn = nn.CrossEntropyLoss( # Loss klasifikasi suhu.
            weight=temp_class_weights
        )

        self.lambda_temp = lambda_temp # bobot MLP Loss

    def forward(self, outputs, yolo_batch, temp_labels):
        """
        1. Menghitung Detection Loss YOLO.
        2. Menjumlahkan seluruh Detection Loss.
        3. Menghitung Temperature Classification Loss.
        4. Menggabungkan kedua loss menjadi Total Loss.

        Input:
            outputs:
                Output model.

                det_out: List feature map detection head

                temp_logits: Logits klasifikasi suhu.
                    Tensor shape [B, num_temp_classes].

            yolo_batch: Ground truth untuk object detection.
                batch_idx:
                    Tensor shape [N].

                cls:
                    Tensor shape [N].

                bboxes:
                    Tensor shape [N,4].

            temp_labels: Label klasifikasi suhu.
                Tensor shape [B].

        Output:
            total_loss: Total loss yang digunakan untuk proses backpropagation.
                Scalar.

            loss_items: Dictionary berisi seluruh komponen loss untuk logging.             
        """
        det_out, temp_logits = outputs # feature map yolo, [B, C]

        # yolo_loss_vec bentuknya [3] = box, cls, dfl
        yolo_loss_vec, yolo_loss_items = self.yolo_loss(det_out, yolo_batch) # GRADIEN, TANPA GRADIEN

        # jadi scalar
        yolo_total_loss = yolo_loss_vec.sum() #

        # weighted loss klasifikasi suhu
        temp_loss = self.temp_loss_fn( 
            temp_logits, #[B, C]
            temp_labels.long()
        )

        # total multitask loss
        total_loss = yolo_total_loss + self.lambda_temp * temp_loss #nantinya akan di .backward()

        loss_items = {
            "total_loss": total_loss.detach(),
            "yolo_total_loss": yolo_total_loss.detach(),
            "box_loss": yolo_loss_items[0].detach(),
            "yolo_cls_loss": yolo_loss_items[1].detach(),
            "dfl_loss": yolo_loss_items[2].detach(),
            "temp_cls_loss": temp_loss.detach(),
        }

        return total_loss, loss_items

In [ ]:
def evaluate_temperature_classification(all_preds, all_labels):
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    results = {
        "accuracy": accuracy_score(all_labels, all_preds),

        "precision_macro": precision_score(
            all_labels,
            all_preds,
            average="macro",
            zero_division=0
        ),

        "recall_macro": recall_score(
            all_labels,
            all_preds,
            average="macro",
            zero_division=0 
        ),

        "f1_macro": f1_score(
            all_labels,
            all_preds,
            average="macro",
            zero_division=0
        ),
    }

    return results

In [ ]:
class YOLOvPredictor(nn.Module):
    def __init__(self, model: nn.Module):
        super().__init__()

        self.device = next(model.parameters()).device

        self.nc = model.nc
        self.reg_max = model.reg_max
        self.no = model.no
        self.stride = model.stride.to(self.device)

        self.decoder = v8DetectionLoss(model)
    
    @torch.no_grad()
    def forward(self, preds):

        parsed = self.decoder.parse_output(preds)

        pred_dist = parsed["boxes"].permute(0, 2, 1).contiguous()
        pred_scores = parsed["scores"].permute(0, 2, 1).contiguous()

        anchor_points, stride_tensor = make_anchors(
            parsed["feats"],
            self.stride,
            0.5,
        )

        pred_boxes = self.decoder.bbox_decode(
            anchor_points,
            pred_dist,
        )

        pred_boxes *= stride_tensor

        return {
            "pred_boxes": pred_boxes,
            "pred_scores": pred_scores.sigmoid(),
        }

In [ ]:
def non_max_suppression(
    pred_boxes,
    pred_scores,
    conf_thres=0.001,
    iou_thres=0.6,
    max_det=300,
):
    outputs = []
    bs = pred_boxes.shape[0]
    max_wh = 7680

    for b in range(bs):

        boxes = pred_boxes[b]
        scores = pred_scores[b]
        conf, cls = scores.max(1)
        keep = conf > conf_thres
        boxes = boxes[keep]
        conf = conf[keep]
        cls = cls[keep]

        if len(boxes) == 0:
            outputs.append({
                "boxes": boxes,
                "scores": conf,
                "labels": cls,
            })
            continue

        offsets = cls.float().unsqueeze(1) * max_wh

        keep_idx = torchvision.ops.nms(
            boxes + offsets,
            conf,
            iou_thres,
        )

        keep_idx = keep_idx[:max_det]

        outputs.append(
            {
                "boxes": boxes[keep_idx],
                "scores": conf[keep_idx],
                "labels": cls[keep_idx],
            }
        )

    return outputs

class Metric:
    def __init__(self):
        self.p = np.array([])
        self.r = np.array([])
        self.f1 = np.array([])
        self.all_ap = np.array([])
        self.ap_class_index = np.array([])
        self.nc = 0

    @property
    def mp(self):
        return self.p.mean() if len(self.p) else 0.0

    @property
    def mr(self):
        return self.r.mean() if len(self.r) else 0.0

    @property
    def map50(self):
        return self.all_ap[:, 0].mean() if len(self.all_ap) else 0.0

    @property
    def map(self):
        return self.all_ap.mean() if len(self.all_ap) else 0.0

    def update(self, results):
        (
            self.p,
            self.r,
            self.f1,
            self.all_ap,
            self.ap_class_index,
            *_
        ) = results
        
def match_predictions(
    pred_classes: torch.Tensor,
    true_classes: torch.Tensor,
    iou: torch.Tensor,
    iouv: torch.Tensor,
) -> torch.Tensor:
    correct = np.zeros(
        (pred_classes.shape[0], iouv.shape[0]),
        dtype=bool,
    )
    correct_class = true_classes[:, None] == pred_classes
    iou = iou * correct_class  # zero out the wrong classes
    iou = iou.cpu().numpy()
    for i, threshold in enumerate(iouv.cpu().tolist()):
        matches = np.nonzero(iou >= threshold)  # IoU > threshold and classes match
        matches = np.array(matches).T
        if matches.shape[0]:
            if matches.shape[0] > 1:
                matches = matches[iou[matches[:, 0], matches[:, 1]].argsort()[::-1]]
                matches = matches[np.unique(matches[:, 1], return_index=True)[1]]
                matches = matches[np.unique(matches[:, 0], return_index=True)[1]]
            correct[matches[:, 1].astype(int), i] = True
    return torch.tensor(correct, dtype=torch.bool, device=pred_classes.device)

def smooth(y: np.ndarray, f: float = 0.05) -> np.ndarray:
    nf = round(len(y) * f * 2) // 2 + 1  # number of filter elements (must be odd)
    p = np.ones(nf // 2)  # ones padding
    yp = np.concatenate((p * y[0], y, p * y[-1]), 0)  # y padded
    return np.convolve(yp, np.ones(nf) / nf, mode="valid")  # y-smoothed

def ap_per_class(
    tp: np.ndarray,
    conf: np.ndarray,
    pred_cls: np.ndarray,
    target_cls: np.ndarray,
    eps: float = 1e-16,
) -> tuple:
    i = np.argsort(-conf)
    tp, conf, pred_cls = tp[i], conf[i], pred_cls[i]
    unique_classes, nt = np.unique(target_cls, return_counts=True)
    nc = unique_classes.shape[0]  # number of classes, number of detections
    x, prec_values = np.linspace(0, 1, 1000), []
    ap, p_curve, r_curve = np.zeros((nc, tp.shape[1])), np.zeros((nc, 1000)), np.zeros((nc, 1000))
    
    for ci, c in enumerate(unique_classes):
        i = pred_cls == c
        n_l = nt[ci]  # number of labels
        n_p = i.sum()  # number of predictions
        if n_p == 0 or n_l == 0:
            continue

        fpc = (1 - tp[i]).cumsum(0)
        tpc = tp[i].cumsum(0)

        recall = tpc / (n_l + eps)  # recall curve
        r_curve[ci] = np.interp(-x, -conf[i], recall[:, 0], left=0)  # negative x, xp because xp decreases
        precision = tpc / (tpc + fpc)  # precision curve
        p_curve[ci] = np.interp(-x, -conf[i], precision[:, 0], left=1)  # p at pr_score

        for j in range(tp.shape[1]):
            ap[ci, j], mpre, mrec = compute_ap(recall[:, j], precision[:, j])
            if j == 0:
                prec_values.append(np.interp(x, mrec, mpre))  # precision at mAP@0.5

    prec_values = np.array(prec_values) if prec_values else np.zeros((1, 1000))  # (nc, 1000)
    f1_curve = 2 * p_curve * r_curve / (p_curve + r_curve + eps)

    i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
    p, r, f1 = p_curve[:, i], r_curve[:, i], f1_curve[:, i]  # max-F1 precision, recall, F1 values
    tp = (r * nt).round()  # true positives
    fp = (tp / (p + eps) - tp).round()  # false positives
    return tp, fp, p, r, f1, ap, unique_classes.astype(int), p_curve, r_curve, f1_curve, x, prec_values

def compute_ap(recall: list[float], precision: list[float]) -> tuple[float, np.ndarray, np.ndarray]:
    mrec = np.concatenate(([0.0], recall, [recall[-1] if len(recall) else 1.0], [1.0]))
    mpre = np.concatenate(([1.0], precision, [0.0], [0.0]))
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))

    method = "interp"  # methods: 'continuous', 'interp'
    if method == "interp":
        x = np.linspace(0, 1, 101)  # 101-point interp (COCO)
        func = np.trapezoid if hasattr(np, "trapezoid") else np.trapz
        ap = func(np.interp(x, mrec, mpre), x)  # integrate
    else:  # 'continuous'
        i = np.where(mrec[1:] != mrec[:-1])[0]  # points where x-axis (recall) changes
        ap = np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])  # area under curve

    return ap, mpre, mrec

class DetectionEvaluator:
    def __init__(self, names):

        self.nc = len(names)
        # IoU threshold
        self.iouv = torch.linspace(0.5, 0.95, 10)
        self.niou = len(self.iouv)
        self.metric = Metric()
        self.reset()

    def reset(self):
        self.stats = {
            "tp": [],
            "conf": [],
            "pred_cls": [],
            "target_cls": []
        }

    def prepare_gt_boxes(self, gt_boxes, image_shape):
        scale = gt_boxes.new_tensor(
            [image_shape[1], image_shape[0],
            image_shape[1], image_shape[0]]
        )
        gt_boxes = torchvision.ops.box_convert(
            gt_boxes,
            "cxcywh",
            "xyxy"
        )
        return gt_boxes * scale
    
    def append_stats(
        self,
        tp,
        conf,
        pred_cls,
        target_cls,
    ):
        self.stats["tp"].append(tp)
        self.stats["conf"].append(conf)
        self.stats["pred_cls"].append(pred_cls)
        self.stats["target_cls"].append(target_cls)

    @torch.no_grad()
    def update(self, preds, batch, image_shape):
        bs = len(preds)
        for si in range(bs):
            pred = preds[si]
            idx = batch["batch_idx"] == si
            gt_cls = batch["cls"][idx].view(-1)
            gt_boxes = batch["bboxes"][idx]

            if len(gt_boxes):
                gt_boxes = self.prepare_gt_boxes(
                    gt_boxes,
                    image_shape
                )
                
            pred_boxes = pred["boxes"]
            pred_scores = pred["scores"]
            pred_cls = pred["labels"]
            tp = np.zeros((0, self.niou), dtype=bool)

            if len(pred_boxes) == 0:
                self.append_stats(
                    tp,
                    pred_scores.cpu().numpy(),
                    pred_cls.cpu().numpy(),
                    gt_cls.cpu().numpy()
                )
                continue

            if len(gt_boxes):
                iou = torchvision.ops.box_iou(
                    gt_boxes,
                    pred_boxes,
                )

                tp = match_predictions(
                        pred_cls,
                        gt_cls,
                        iou,
                        self.iouv,
                    ).cpu().numpy()

            else:
                tp = np.zeros((len(pred_boxes), self.niou), dtype=bool)

            self.append_stats(
                np.asarray(tp),
                pred_scores.cpu().numpy(),
                pred_cls.cpu().numpy(),
                gt_cls.cpu().numpy(),
            )

    def compute(self):
        stats = {}
        for k, v in self.stats.items():
            if len(v):
                stats[k] = np.concatenate(v, axis=0)
                
            else:
                stats[k] = np.array([])

        if len(stats["tp"]) == 0:
            return {
                "precision": 0,
                "recall": 0,
                "map50": 0,
                "map50_95": 0,
            }

        results = ap_per_class(
            tp=stats["tp"],
            conf=stats["conf"],
            pred_cls=stats["pred_cls"],
            target_cls=stats["target_cls"],
        )[2:]

        self.metric.nc = self.nc
        self.metric.update(results)

        return {
            "precision": float(self.metric.mp),
            "recall": float(self.metric.mr),
            "map50": float(self.metric.map50),
            "map50_95": float(self.metric.map),
        }

In [ ]:
def plot_metric(train_values, val_values, ylabel, title, filename):
    plt.figure(figsize=(8, 5))
    plt.plot(train_values, marker="o", label="Train")
    plt.plot(val_values, marker="o", label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.savefig(filename)
    plt.show()

def plot_single_metric(values, ylabel, title, filename):
    plt.figure(figsize=(8,5))
    plt.plot(values, marker='o')
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True)
    plt.savefig(filename)
    plt.show()

In [ ]:
def build_model():
    my_model = YOLOv12(nc=3)
    model_dict = my_model.state_dict() 
    
    official_model = YOLO("yolo12x.pt") 
    official_dict = official_model.state_dict() 
    new_dict = {} 
    loaded_pairs = [] 

    for (custom_key, custom_value), (official_key, official_value) in zip(model_dict.items(),official_dict.items()):
        if custom_value.shape == official_value.shape:
            new_dict[custom_key] = official_value
            loaded_pairs.append((custom_key, official_key, tuple(custom_value.shape)))
                
        else:
            print(
                f"SKIP | custom: {custom_key} {tuple(custom_value.shape)} "
                f"!= official: {official_key} {tuple(official_value.shape)}"
            )

    missing, unexpected = my_model.load_state_dict(new_dict, strict=False)
                        
    print(f"Loaded keys: {len(new_dict)} / {len(model_dict)}")
    print(f"Missing keys: {len(missing)}")
    print(f"Unexpected keys: {len(unexpected)}")        

    # Freeze Backbone 
    for param in my_model.backbone.parameters(): 
        param.requires_grad = False 
    
    return my_model

In [ ]:
def train_one_config(config):
    best_model_state = None
    my_model = build_model().to(device)
    predictor = YOLOvPredictor(my_model)
    evaluator = DetectionEvaluator(
        names={0: "arm", 1: "head", 2:"upper_body"}
    )
    temp_class_counts = torch.tensor(
        [286, 365, 541],
        dtype=torch.float32
    )
    
    num_temp_classes = len(temp_class_counts)
    total_temp_samples = temp_class_counts.sum()
    # total sample / (jumlah kelas * jumlah sampel pada setiap kelas)
    temp_class_weights = total_temp_samples / (num_temp_classes * temp_class_counts)
    print("Temperature class weights:", temp_class_weights)
    
    criterion = MultiTaskLoss (my_model, lambda_temp=1.0, temp_class_weights=temp_class_weights)
    train_loader = DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=True, collate_fn=collate_fn, drop_last=True, num_workers=0, pin_memory=True ) 
    val_loader = DataLoader(val_dataset, batch_size=config["batch_size"], shuffle=False, collate_fn=collate_fn, drop_last=False, num_workers=0, pin_memory=True )

    if config["optimizer"] == "AdamW":
        optimizer = optim.AdamW(
            filter(lambda p: p.requires_grad, my_model.parameters()),
            lr=config["lr"]
        )
    elif config["optimizer"] == "SGD":
        optimizer = optim.SGD(
            filter(lambda p: p.requires_grad, my_model.parameters()),
            lr=config["lr"],
            momentum=0.9
        )
    elif config["optimizer"] == "RMSprop":
        optimizer = optim.RMSprop(
            filter(lambda p: p.requires_grad, my_model.parameters()),
            lr=config["lr"]
        )
    num_epochs = 100
    train_losses = []
    val_losses = []
    epoch_logs = []
    best_val_loss = float("inf")
    best_metrics = None
    best_epoch = 0
    train_box_losses = []
    val_box_losses = []

    train_cls_losses = []
    val_cls_losses = []

    train_dfl_losses = []
    val_dfl_losses = []

    train_temp_losses = []
    val_temp_losses = []

    accuracies = []
    precisions = []
    recalls = []
    f1_scores = []
    map50_scores = []
    map5095_scores = []
    
    
    for epoch in range(num_epochs): 
        my_model.train()
        total_train_loss = 0
        total_train_box_loss = 0
        total_train_yolo_cls_loss = 0
        total_train_temp_cls_loss = 0
        total_train_dfl_loss = 0
        
        for images, targets, temp_labels in train_loader:

            images = images.to(device)
            temp_labels = temp_labels.to(device)
            batch_targets = build_targets(targets, device)

            optimizer.zero_grad()

            outputs = my_model(images)

            loss, loss_items = criterion(outputs, batch_targets, temp_labels)

            loss.backward()
            optimizer.step()
                
            total_train_box_loss += loss_items["box_loss"].item()
            total_train_yolo_cls_loss += loss_items["yolo_cls_loss"].item()
            total_train_temp_cls_loss += loss_items["temp_cls_loss"].item()
            total_train_dfl_loss += loss_items["dfl_loss"].item()
            total_train_loss += loss.item()
            
        avg_train_box_loss = total_train_box_loss / len(train_loader)
        avg_train_yolo_cls_loss = total_train_yolo_cls_loss / len(train_loader)
        avg_train_temp_cls_loss = total_train_temp_cls_loss / len(train_loader)
        avg_train_loss = total_train_loss / len(train_loader)
        avg_train_dfl_loss = total_train_dfl_loss / len(train_loader)
        
        my_model.eval()
        total_val_loss = 0
        total_val_box_loss = 0
        total_val_yolo_cls_loss = 0
        total_val_temp_cls_loss = 0
        total_val_dfl_loss = 0
        
        all_temp_preds = []
        all_temp_labels = []
        
        evaluator.reset()
        with torch.no_grad():
            for images, targets, temp_labels in val_loader:

                images = images.to(device)
                temp_labels = temp_labels.to(device)

                outputs = my_model(images)

                batch_targets = build_targets(targets, device)

                loss, loss_items = criterion(outputs, batch_targets, temp_labels)
                    
                det_out, temp_logits = outputs
                temp_preds = torch.argmax(temp_logits, dim=1)

                decoded = predictor(det_out)

                preds = non_max_suppression(
                    decoded["pred_boxes"],
                    decoded["pred_scores"],
                    conf_thres=0.001,
                    iou_thres=0.6,
                )

                evaluator.update(
                    preds,
                    batch_targets,
                    images.shape[2:],   # (H, W)
                )
                
                all_temp_preds.extend(temp_preds.cpu().numpy())
                all_temp_labels.extend(temp_labels.cpu().numpy())
                    
                total_val_loss += loss.item()
                total_val_box_loss += loss_items["box_loss"].item()
                total_val_yolo_cls_loss += loss_items["yolo_cls_loss"].item()
                total_val_temp_cls_loss += loss_items["temp_cls_loss"].item()
                total_val_dfl_loss += loss_items["dfl_loss"].item()

        result = evaluator.compute()

        map50 = result["map50"]
        map5095 = result["map50_95"]

        avg_val_loss = total_val_loss / len(val_loader)
        avg_val_box_loss = total_val_box_loss / len(val_loader)
        avg_val_yolo_cls_loss = total_val_yolo_cls_loss / len(val_loader)
        avg_val_temp_cls_loss = total_val_temp_cls_loss / len(val_loader)
        avg_val_dfl_loss = total_val_dfl_loss / len(val_loader)

        temp_metrics = evaluate_temperature_classification(
            all_preds=all_temp_preds,
            all_labels=all_temp_labels
        )
            
        # Simpan loss setiap epoch
        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)

        epoch_logs.append({
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss,

            "train_box_loss": avg_train_box_loss,
            "train_yolo_cls_loss": avg_train_yolo_cls_loss,
            "train_temp_cls_loss": avg_train_temp_cls_loss,
            "train_dfl_loss": avg_train_dfl_loss,

            "val_box_loss": avg_val_box_loss,
            "val_yolo_cls_loss": avg_val_yolo_cls_loss,
            "val_temp_cls_loss": avg_val_temp_cls_loss,
            "val_dfl_loss": avg_val_dfl_loss,    

            "val_temp_acc": temp_metrics["accuracy"],
            "val_temp_precision_macro": temp_metrics["precision_macro"],
            "val_temp_recall_macro": temp_metrics["recall_macro"],
            "val_temp_f1_macro": temp_metrics["f1_macro"],
            "map50": map50,
            "map50_95": map5095,
        })

        print(
            f"Epoch [{epoch+1}/{num_epochs}] "
            f"Train Loss: {avg_train_loss:.4f} "
            f"Val Loss: {avg_val_loss:.4f} | "
            f"Train Box: {avg_train_box_loss:.4f} "
            f"Train YOLO Cls: {avg_train_yolo_cls_loss:.4f} "
            f"Train DFL: {avg_train_dfl_loss:.4f} "
            f"Train Temp Cls: {avg_train_temp_cls_loss:.4f} | "
            f"Val Box: {avg_val_box_loss:.4f} "
            f"Val YOLO Cls: {avg_val_yolo_cls_loss:.4f} "
            f"Val DFL: {avg_val_dfl_loss:.4f} "
            f"Val Temp Cls: {avg_val_temp_cls_loss:.4f} | "
            f"Val Temp Acc: {temp_metrics['accuracy']:.4f} "
            f"Precision: {temp_metrics['precision_macro']:.4f} "
            f"Recall: {temp_metrics['recall_macro']:.4f} "
            f"F1: {temp_metrics['f1_macro']:.4f} | "
            f"mAP50: {map50:.4f} | "
            f"mAP50_95: {map5095:.4f}"
        )
        
        train_box_losses.append(avg_train_box_loss)
        val_box_losses.append(avg_val_box_loss)

        train_cls_losses.append(avg_train_yolo_cls_loss)
        val_cls_losses.append(avg_val_yolo_cls_loss)

        train_dfl_losses.append(avg_train_dfl_loss)
        val_dfl_losses.append(avg_val_dfl_loss)

        train_temp_losses.append(avg_train_temp_cls_loss)
        val_temp_losses.append(avg_val_temp_cls_loss)

        accuracies.append(temp_metrics["accuracy"])
        precisions.append(temp_metrics["precision_macro"])
        recalls.append(temp_metrics["recall_macro"])
        f1_scores.append(temp_metrics["f1_macro"])
        
        map50_scores.append(map50)
        map5095_scores.append(map5095)
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_epoch = epoch+1
            best_metrics = {
                "accuracy": temp_metrics["accuracy"],
                "precision_macro": temp_metrics["precision_macro"],
                "recall_macro": temp_metrics["recall_macro"],
                "f1_macro": temp_metrics["f1_macro"],
                "map50": map50,
                "map50_95": map5095,
            }
            best_model_state = copy.deepcopy(my_model.state_dict())

            torch.save({    
                "model_state_dict": best_model_state,
                "config": config,
                "best_epoch": best_epoch,
                "best_val_loss": best_val_loss,
                "metrics": best_metrics,
                }, f"best_model_{config['optimizer']}_lr{config['lr']}_bs{config['batch_size']}.pth"
            )
    
    plot_metric(
        train_losses,
        val_losses,
        ylabel="Loss",
        title="Total Loss",
        filename=f"total_loss_{config['optimizer']}_lr{config['lr']}_bs{config['batch_size']}.png"
    )

    plot_metric(
        train_box_losses,
        val_box_losses,
        ylabel="Box Loss",
        title="Bounding Box Loss",
        filename=f"BoxLoss{config['optimizer']}_lr{config['lr']}_bs{config['batch_size']}.png"
    )

    plot_metric(
        train_dfl_losses,
        val_dfl_losses,
        ylabel="DFL Loss",
        title="DFL Loss",
        filename=f"DFL_Loss{config['optimizer']}_lr{config['lr']}_bs{config['batch_size']}.png"
    )

    plot_metric(
        train_cls_losses,
        val_cls_losses,
        ylabel="Classification Loss",
        title="YOLO Classification Loss",
        filename=f"YOLO_CLS_Loss{config['optimizer']}_lr{config['lr']}_bs{config['batch_size']}.png"
    )

    plot_metric(
        train_temp_losses,
        val_temp_losses,
        ylabel="Temperature Classification Loss",
        title="Temperature Classification Loss",
        filename=f"MLP_Loss{config['optimizer']}_lr{config['lr']}_bs{config['batch_size']}.png"
    )

    plot_single_metric(
        accuracies,
        "Accuracy",
        "Validation Accuracy",
        filename=f"Accuracy{config['optimizer']}_lr{config['lr']}_bs{config['batch_size']}.png"
    )

    plot_single_metric(
        precisions,
        "Precision",
        "Validation Precision",
        filename=f"Precision{config['optimizer']}_lr{config['lr']}_bs{config['batch_size']}.png"
    )

    plot_single_metric(
        recalls,
        "Recall",
        "Validation Recall",
        filename=f"recalls{config['optimizer']}_lr{config['lr']}_bs{config['batch_size']}.png"
    )

    plot_single_metric(
        f1_scores,
        "F1 Score",
        "Validation F1 Score",      
        filename=f"f1_scores{config['optimizer']}_lr{config['lr']}_bs{config['batch_size']}.png"
    )

    plot_single_metric(
        map50_scores,
        ylabel="mAP@0.50",
        title="Detection mAP@0.5",
        filename=f"mAP@50{config['optimizer']}_lr{config['lr']}_bs{config['batch_size']}.png"
    )
    
    plot_single_metric(
        map5095_scores,
        ylabel="mAP@0.50:0.95",
        title="Detection mAP@0.50:0.95",
        filename=f"mAP@0.5:0.95{config['optimizer']}_lr{config['lr']}_bs{config['batch_size']}.png"
    )
            
    df= pd.DataFrame(epoch_logs)
    csv_name = (
        f'training_log_{config["optimizer"]}_lr{config["lr"]}_bs{config["batch_size"]}.csv'
    )
    df.to_csv(csv_name, index=False)
                            
    return {
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "best_metrics": best_metrics,
        "config": config,
    }

In [ ]:
baseline = {
    "lr": 0.01,
    "batch_size": 16,
    "optimizer": "AdamW"
}

hyperparams = {
    "lr": [0.01,0.001],
    "batch_size": [16,32],
    "optimizer": ["AdamW","RMSprop", "SGD"]
}

stages = [
    "lr",
    "batch_size",
    "optimizer"
]

best_config = baseline.copy()
best_loss = float("inf")
experiment_results = [] 

## TRAINING 
baseline_result = train_one_config(baseline)

experiment_results.append({
    "stage": "baseline",
    "lr": baseline["lr"],
    "batch_size": baseline["batch_size"],
    "optimizer": baseline["optimizer"],
    "best_val_loss": baseline_result["best_val_loss"],
    "best_epoch": baseline_result["best_epoch"],
    "accuracy": baseline_result["best_metrics"]["accuracy"],
    "precision": baseline_result["best_metrics"]["precision_macro"],
    "recall": baseline_result["best_metrics"]["recall_macro"],
    "f1": baseline_result["best_metrics"]["f1_macro"],
    "map50": baseline_result["best_metrics"]["map50"], 
    "map50_95": baseline_result["best_metrics"]["map50_95"], 
})

best_loss = baseline_result["best_val_loss"]
best_config = baseline.copy()

for stage in stages:

    print("="*60)
    print(f"Testing {stage}")

    for value in hyperparams[stage]:
        if value == best_config[stage]:
            continue
        
        config = best_config.copy()
        config[stage] = value
        
        print(config)
        result = train_one_config(config) 
        
        print("="*60)
        print(f"Stage      : {stage}")
        print(f"Config     : {config}")
        print(f"Val Loss   : {result['best_val_loss']:.4f}")
        print(f"Best Loss  : {best_loss:.4f}")

        if result["best_val_loss"] < best_loss:
            best_loss = result["best_val_loss"]
            best_config = config.copy()
            print("--New Best Configuration--")
        
        experiment_results.append({
            "stage": stage,
            "lr": config["lr"],
            "batch_size": config["batch_size"],
            "optimizer": config["optimizer"],
            "best_val_loss": result["best_val_loss"],
            "best_epoch": result["best_epoch"],
            "accuracy": result["best_metrics"]["accuracy"],
            "precision": result["best_metrics"]["precision_macro"],
            "recall": result["best_metrics"]["recall_macro"],
            "f1": result["best_metrics"]["f1_macro"],
            "map50": result["best_metrics"]["map50"], 
            "map50_95": result["best_metrics"]["map50_95"], 
        })
            
print("="*70)
print("FINAL RESULT")

print(best_config)
print(best_loss)
pd.DataFrame(experiment_results)